# 01 — Data Cleaning & Feature Engineering

This notebook loads the raw Facebook analytics export, cleans and validates it,
engineers all derived features, and exports a single clean dataset for use in all
subsequent notebooks.

**Output:** `data/clean_posts.csv`

**Sections:**
1. Configuration
2. Data Loading
3. Initial Inspection
4. Data Cleaning
5. Missing Value Analysis
6. Feature Engineering
   - 6.1 Timezone Correction & Verification
   - 6.2 Post Origin
   - 6.3 Text Length
   - 6.4 Structural Text Features
   - 6.5 Engagement Rate
   - 6.6 View Rates
   - 6.7 Post Category (4-Quadrant)
7. Dataset Summary
8. Export

## 1. Configuration

**Update this cell whenever you add new data.**
All other cells reference these variables — nothing else needs to change.

In [46]:
from pathlib import Path

# ── Data ────────────────────────────────────────────────────────────────────
DATA_FILE      = Path("../data/010625_190326_fb_posts.csv")  # raw CSV from Facebook
OUTPUT_FILE    = Path("../data/clean_posts.csv")             # clean export path

# ── Page identity ───────────────────────────────────────────────────────────
MY_PAGE_NAME   = "Liz Izakson Mashal"   # your Facebook page name

# ── Timezone correction ─────────────────────────────────────────────────────
# Facebook exports timestamps with an inconsistent UTC offset.
# Two offsets are needed because the offset changed between export batches.
# Verify each time new data is added (see Section 6.1).
TZ_CUTOFF      = "2026-03-08"   # date where the offset changes
TZ_OFFSET_OLD  = 10             # hours to add for posts BEFORE the cutoff
TZ_OFFSET_NEW  = 9              # hours to add for posts FROM the cutoff onwards

# ── Freshness check ─────────────────────────────────────────────────────────
# Update to the latest post date every time you add new data.
# This triggers a warning if new posts are detected that need timezone verification.
KNOWN_LATEST   = "2026-03-19"

## 2. Data Loading

Import libraries and load the raw Facebook analytics CSV.

In [47]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_FILE)

print(f"Loaded {len(df)} rows from {DATA_FILE.name}")
df.head()

Loaded 137 rows from 010625_190326_fb_posts.csv


,Post ID,Page ID,Page name,Title,Duration (sec),Publish time,Permalink,Post type,Data comment,Date,...,Impressions,Interactions,Reactions,Saves,Shares,Views,Viewers,Net follows,Average Seconds viewed,Seconds viewed
0,10234810932441995,1461428297,Liz Izakson Mashal,חופשה עם ילדים זאת מעין חצי חופשה. \n\nלא מקבל...,0,08/13/2025 22:01,https://www.facebook.com/liz.izakson/posts/pfb...,Photo,NaN,Lifetime,...,945.0,53.0,48.0,0.0,1.0,1751,789.0,NaN,NaN,NaN
1,10238066919799644,1461428297,Liz Izakson Mashal,מלחמה וסטרס עושות משהו מעניין לזמן.\n\nהן מותח...,0,03/18/2026 22:45,https://www.facebook.com/liz.izakson/posts/pfb...,Photo,NaN,Lifetime,...,457.0,48.0,36.0,0.0,0.0,831,433.0,NaN,NaN,NaN
2,10237996040547707,1461428297,Liz Izakson Mashal,מה אנחנו באמת רוצים כשאנחנו חושקים במישהו?\n\n...,0,03/15/2026 22:30,https://www.facebook.com/liz.izakson/posts/pfb...,Photo,NaN,Lifetime,...,5340.0,62.0,30.0,3.0,0.0,7107,4905.0,NaN,NaN,NaN
3,10237942817737170,1461428297,Liz Izakson Mashal,"בימים מטושטשים ולא ברורים כאלה,\nאני אוהבת בעי...",0,03/13/2026 07:19,https://www.facebook.com/liz.izakson/posts/pfb...,Photo,NaN,Lifetime,...,414.0,21.0,19.0,0.0,0.0,572,341.0,NaN,NaN,NaN
4,10237913037112673,1461428297,Liz Izakson Mashal,מפגש עם ספר הוא לפעמים גם מפגש בין שני אנשים ש...,0,03/11/2026 23:07,https://www.facebook.com/liz.izakson/posts/pfb...,Content,NaN,Lifetime,...,211.0,17.0,11.0,0.0,0.0,307,176.0,NaN,NaN,NaN


## 3. Initial Inspection

A quick look at column types and null counts to understand what we're working with
before making any changes.

In [48]:
print(f"Posts: {len(df)}")
print(f"Variables: {len(df.columns)}")
print()
df.info()

Posts: 137
Variables: 22

<class 'pandas.DataFrame'>
RangeIndex: 137 entries, 0 to 136
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Post ID                 137 non-null    int64  
 1   Page ID                 137 non-null    int64  
 2   Page name               137 non-null    str    
 3   Title                   135 non-null    str    
 4   Duration (sec)          137 non-null    int64  
 5   Publish time            137 non-null    str    
 6   Permalink               137 non-null    str    
 7   Post type               137 non-null    str    
 8   Data comment            0 non-null      float64
 9   Date                    137 non-null    str    
 10  Comments                100 non-null    float64
 11  Distribution            99 non-null     str    
 12  Impressions             99 non-null     float64
 13  Interactions            100 non-null    float64
 14  Reactions               100

## 4. Data Cleaning

Three steps:
1. Normalise column names to lowercase with underscores — makes them easier to type and avoids bugs from spaces or special characters.
2. Coerce all metric columns to numeric — the CSV sometimes stores numbers as strings; `errors='coerce'` turns unparseable values into `NaN` rather than crashing.
3. Remove cover photo changes — these are not content posts and would bias any analysis of what makes written posts resonate.

In [49]:
# Normalise column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]+", "_", regex=True)
)

print("Column names after cleaning:")
print(df.columns.tolist())

Column names after cleaning:
['post_id', 'page_id', 'page_name', 'title', 'duration_sec_', 'publish_time', 'permalink', 'post_type', 'data_comment', 'date', 'comments', 'distribution', 'impressions', 'interactions', 'reactions', 'saves', 'shares', 'views', 'viewers', 'net_follows', 'average_seconds_viewed', 'seconds_viewed']


In [50]:
# Coerce metric columns to numeric
numeric_cols = [
    "views", "comments", "impressions", "interactions",
    "reactions", "saves", "shares", "viewers",
    "average_seconds_viewed", "seconds_viewed",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Post type distribution:")
print(df["post_type"].value_counts())

Post type distribution:
post_type
Photo      108
Content     20
Reel         7
Link         2
Name: count, dtype: int64


## 5. Missing Value Analysis

Not all missing values are errors — some are structural (a metric simply doesn't apply
to a post type). Understanding *why* values are missing determines whether to drop,
keep, or flag them.

Three patterns exist in this dataset:

| Group | Missing columns | Reason | Action |
|---|---|---|---|
| **Tagged posts** (other pages) | `viewers`, `impressions`, `interactions` | Facebook doesn't report these metrics for posts from other pages | Remove — Section 6.2 |
| **Liz's posts — missing `impressions`** | `impressions` | Too recent or partial export | Drop |
| **Liz's older posts — missing `viewers` only** | `viewers` | Column not available in older exports | Keep — other metrics present |
| **Reel-only metrics** | `average_seconds_viewed`, `seconds_viewed` | Video metrics — not applicable to photos/text | Keep as structural NaN |

In [51]:
# Missing value count per metric column
print("Missing values per column:")
print(df[numeric_cols].isna().sum())

Missing values per column:
views                       0
comments                   37
impressions                38
interactions               37
reactions                  37
saves                      37
shares                     37
viewers                    43
average_seconds_viewed    130
seconds_viewed            130
dtype: int64


In [52]:
# Confirm that video-only metrics are exclusively populated for Reels
# (missingness in other types is structural, not a data quality issue)
df.groupby("post_type")[["average_seconds_viewed", "seconds_viewed"]].count()

,average_seconds_viewed,seconds_viewed
post_type,,
Content,0,0
Link,0,0
Photo,0,0
Reel,7,7


In [53]:
# Drop Liz's own posts that are missing impressions
# These are either too recent for Facebook to have finalised metrics,
# or duplicates from a partial export.
# Dropped posts are printed below for manual verification.

mask_drop = (df["page_name"] == MY_PAGE_NAME) & df["impressions"].isna()

if mask_drop.sum() > 0:
    print(f"Posts to be dropped ({mask_drop.sum()}):")
    display(
        df[mask_drop][["publish_time", "post_type", "title"]]
        .sort_values("publish_time", ascending=False)
        .reset_index(drop=True)
    )
else:
    print("No posts with missing impressions found.")

before = len(df)
df = df[~mask_drop].copy()
print(f"\nRemoved {before - len(df)} post(s) — {len(df)} remain.")

Posts to be dropped (1):


,publish_time,post_type,title
0,03/04/2026 09:50,Content,אתמול כתבתי כאן על החוויות ממפגש החתימות במודי...



Removed 1 post(s) — 136 remain.


### Removing Cover Photo Changes

Cover photo changes are excluded from the analysis for two reasons:

1. **No text content** — they have no title or written text, making them incompatible
   with any linguistic analysis in later notebooks.
2. **Different algorithmic behavior** — Facebook automatically notifies followers when
   a cover photo changes, generating engagement through a push notification mechanic
   rather than through organic feed distribution. This means their engagement metrics
   reflect Facebook's notification system, not audience response to content — which is
   precisely what this project aims to study.

Including them would bias any model that tries to learn what makes a written post resonate.

In [ ]:
# Remove cover photo changes — excluded for two reasons:
# 1. No text content — incompatible with linguistic analysis
# 2. Different algorithmic mechanic — engagement is driven by Facebook's
#    automatic follower notifications, not organic feed distribution

before = len(df)
cover_photos = df[df["title"].isna() | (df["title"].str.strip() == "")]

if len(cover_photos) > 0:
    print(f"Cover photo posts to be removed ({len(cover_photos)}):")
    display(
        cover_photos[["publish_time", "post_type", "views", "interactions"]]
        .reset_index(drop=True)
    )
else:
    print("No cover photo posts found.")

df = df[df["title"].notna() & (df["title"].str.strip() != "")].copy()
print(f"\nRemoved {before - len(df)} cover photo post(s) — {len(df)} remain.")

## 6. Feature Engineering

All new columns added here are derived directly from the raw data using arithmetic or
rule-based logic — no interpretation, no charts. The resulting dataframe is what gets
exported and used in all other notebooks.

### 6.1 Timezone Correction & Verification

Facebook's analytics export uses UTC timestamps that do not match the actual local
posting time. The offset has changed between export batches, so two correction values
are needed (configured in Section 1).

**Always verify the corrected times manually** before proceeding — check that the
times shown below match when you remember actually posting.

In [54]:
# Parse raw timestamps
df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce")

# Apply the two-phase timezone correction using the cutoff from configuration
cutoff = pd.Timestamp(TZ_CUTOFF)
mask_old = df["publish_time"] < cutoff

df.loc[ mask_old, "publish_time"] += pd.Timedelta(hours=TZ_OFFSET_OLD)
df.loc[~mask_old, "publish_time"] += pd.Timedelta(hours=TZ_OFFSET_NEW)

# Extract temporal features
df["date"]    = df["publish_time"].dt.date
df["time"]    = df["publish_time"].dt.time
df["hour"]    = df["publish_time"].dt.hour
df["weekday"] = df["publish_time"].dt.day_name()

In [55]:
# ── MANUAL VERIFICATION ──────────────────────────────────────────────────────
# Check the 10 most recent posts. Do the corrected times match when you
# actually remember posting? If not, update TZ_OFFSET_NEW in Section 1.

known_latest = pd.Timestamp(KNOWN_LATEST)
new_posts = df[df["publish_time"] > known_latest]

if not new_posts.empty:
    print(f"⚠️  {len(new_posts)} new post(s) detected beyond KNOWN_LATEST ({KNOWN_LATEST}).")
    print("Verify their corrected times below, then update KNOWN_LATEST in Section 1.\n")

print("10 most recent posts after timezone correction — verify these look correct:")
(
    df.sort_values("publish_time", ascending=False)
    [["date", "time", "weekday", "title"]]
    .head(10)
)

⚠️  1 new post(s) detected beyond KNOWN_LATEST (2026-03-19).
Verify their corrected times below, then update KNOWN_LATEST in Section 1.

10 most recent posts after timezone correction — verify these look correct:


,date,time,weekday,title
1,2026-03-19,07:45:00,Thursday,מלחמה וסטרס עושות משהו מעניין לזמן.\n\nהן מותח...
2,2026-03-16,07:30:00,Monday,מה אנחנו באמת רוצים כשאנחנו חושקים במישהו?\n\n...
3,2026-03-13,16:19:00,Friday,"בימים מטושטשים ולא ברורים כאלה,\nאני אוהבת בעי..."
4,2026-03-12,08:07:00,Thursday,מפגש עם ספר הוא לפעמים גם מפגש בין שני אנשים ש...
5,2026-03-10,07:45:00,Tuesday,"יש מקומות שנשארים חרוטים בגופינו.\n\nמסדרונות,..."
6,2026-03-08,19:15:00,Sunday,הדבר שהכי מייצג אישה עבורי הוא שינויים. \n\nשי...
7,2026-03-07,19:19:00,Saturday,מה שמרגיע אותי בתוך הכאוס:\n\n1. לבהות בתווי ה...
8,2026-03-05,15:53:00,Thursday,"המלצת קריאה:""בין המסדרונות""-ליז איזקסון משל. ה..."
10,2026-03-04,19:49:00,Wednesday,"⁨ ⁨ רגע לפני שהמלחמה התחילה והכל נסגר,\nוהתכנס..."
11,2026-03-03,19:15:00,Tuesday,"רגע לפני שהתחילה עוד מלחמה שסוגרת אותנו בבתים,..."


### 6.2 Post Origin

Posts can appear on your page in two ways:
- **My Post** — you posted it directly (original content or a share you made)
- **Tagged** — someone else tagged you, so it appeared on your page

Tagged posts come from other pages and Facebook doesn't report full metrics for them,
so they are separated out and excluded from the main analysis.

**Note on cover photo posts:** A small number of posts have an empty `title`.
These are cover photo changes — Facebook doesn't assign text to those posts.
They are kept in the dataset but worth noting when they appear in rankings.

In [56]:
df["post_origin"] = df["page_name"].apply(
    lambda x: "My Post" if x == MY_PAGE_NAME else "Tagged"
)

df_tagged = df[df["post_origin"] == "Tagged"].copy()
df        = df[df["post_origin"] == "My Post"].copy()

print(f"My posts (main analysis): {len(df)}")
print(f"Tagged by others (excluded): {len(df_tagged)}")
print()
print("Post type breakdown:")
print(df["post_type"].value_counts())

My posts (main analysis): 99
Tagged by others (excluded): 37

Post type breakdown:
post_type
Photo      77
Content    15
Reel        7
Name: count, dtype: int64


### 6.3 Text Length

A simple count of characters in the post title (the text content of the post).
This is the most basic linguistic feature and requires no external tools.
Posts with no title (cover photo changes) get a length of 0.

In [ ]:
df["text_length_chars"] = df["title"].fillna("").str.len()
df["text_length_words"] = df["title"].fillna("").str.split().str.len()

print("Text length summary:")
print()
print("Characters:")
print(df["text_length_chars"].describe().round(1))
print()
print("Words:")
print(df["text_length_words"].describe().round(1))

Text length summary (characters):
count      99.0
mean      998.5
std       658.1
min         0.0
25%       543.5
50%       862.0
75%      1436.5
max      2837.0
Name: text_length, dtype: float64


### 6.4 Structural Text Features

Features that can be extracted directly from the raw text without requiring
an LLM — hashtag usage, tagging of other users, and keyword-based detection
of children mentions. These are cheaper and more reliable to detect via
pattern matching than through language model annotation.

- **has_hashtag** — does the post contain at least one hashtag (#)?
- **hashtag_count** — how many hashtags does the post contain?
- **tag_count** — how many people or pages are tagged in this post?
  (detected via LLM annotation in `03_text_analysis.ipynb` — Facebook exports
  do not use @ syntax so this cannot be detected via pattern matching)
- **mentions_children** — does the post contain keywords related to the author's children
  (דניאל, נועם, ילדים, etc.)? This is a surface-level flag — deeper parenthood theme
  analysis is handled in `03_text_analysis.ipynb` via LLM annotation.

In [ ]:
# Hashtag features
df["has_hashtag"]   = df["title"].str.contains("#", na=False).astype(int)
df["hashtag_count"] = df["title"].str.count("#")

# Tagging feature

# Children mention — keyword-based surface signal
CHILDREN_KEYWORDS = ["ילדים", "דניאל", "נועם", "הבן", "הילד", "הילדה", "ילד"]
df["mentions_children"] = df["title"].apply(
    lambda x: int(any(kw in str(x) for kw in CHILDREN_KEYWORDS))
)

print("Structural feature summary:")
print(f"  Posts with hashtags:      {df['has_hashtag'].sum()} ({df['has_hashtag'].mean():.0%})")
print(f"  Avg hashtags per post:    {df['hashtag_count'].mean():.2f}")
print(f"  Posts mentioning children:{df['mentions_children'].sum()} ({df['mentions_children'].mean():.0%})")

### 6.5 Engagement Rate

`engagement_rate = interactions / impressions`

Normalising by impressions makes posts with very different reach levels comparable —
a post seen by 100 people that got 10 interactions (10%) is more engaging than one
seen by 5,000 people that got 100 interactions (2%).

Posts with 0 impressions are set to `NaN` to avoid division by zero.

In [58]:
df["engagement_rate"] = df["interactions"] / df["impressions"].replace(0, np.nan)

print("Engagement rate summary:")
print(df["engagement_rate"].describe().round(4))

Engagement rate summary:
count    99.0000
mean      0.1159
std       0.3614
min       0.0092
25%       0.0516
50%       0.0660
75%       0.0806
max       3.3333
Name: engagement_rate, dtype: float64


### 6.6 View Rates

Two additional derived metrics that describe how people interact with posts at the
viewing level — before they react or comment:

- **`repeat_view_rate`** = `views / viewers` — how many times on average each unique
  viewer looked at the post. A value above 1 means people returned to view it more than once.
- **`view_through_rate`** = `viewers / impressions` — of everyone who saw the post in
  their feed, what share actually stopped to view it (versus scrolling past).

Both use `.replace(0, NaN)` to avoid division by zero.

In [59]:
df["repeat_view_rate"]  = df["views"]   / df["viewers"].replace(0, np.nan)
df["view_through_rate"] = df["viewers"] / df["impressions"].replace(0, np.nan)

print("View rate summary by post type:")
df.groupby("post_type")[["views", "viewers", "repeat_view_rate", "view_through_rate"]].mean().round(2)

View rate summary by post type:


,views,viewers,repeat_view_rate,view_through_rate
post_type,,,,
Content,466.47,250.67,1.83,0.85
Photo,2145.64,1202.01,1.97,0.83
Reel,873.00,592.29,1.54,0.79


### 6.7 Post Category (4-Quadrant)

Each post is placed into one of four categories based on whether its impressions
and engagement rate are above or below the dataset median:

| | High Engagement Rate | Low Engagement Rate |
|---|---|---|
| **High Impressions** | **Viral** — widely seen *and* resonated | **Algorithm Pushed** — wide reach, low resonance |
| **Low Impressions** | **Audience Favorite** — limited reach but loved by those who saw it | **Low Performance** — limited reach and limited resonance |

Medians are recalculated each run, so the categories automatically rebalance
as new data is added.

In [60]:
impressions_median = df["impressions"].median()
engagement_median  = df["engagement_rate"].median()

def classify_post(row):
    high_imp = row["impressions"]    >= impressions_median
    high_eng = row["engagement_rate"] >= engagement_median
    if   high_imp and     high_eng: return "Viral"
    elif high_imp and not high_eng: return "Algorithm Pushed"
    elif not high_imp and high_eng: return "Audience Favorite"
    else:                           return "Low Performance"

df["post_category"] = df.apply(classify_post, axis=1)

print("Posts per category:")
print(df["post_category"].value_counts())
print()
print("Category breakdown by post type:")
pd.crosstab(df["post_category"], df["post_type"])

Posts per category:
post_category
Algorithm Pushed     27
Audience Favorite    27
Viral                23
Low Performance      22
Name: count, dtype: int64

Category breakdown by post type:


post_type,Content,Photo,Reel
post_category,,,
Algorithm Pushed,0,26,1
Audience Favorite,10,13,4
Low Performance,5,15,2
Viral,0,23,0


## 7. Dataset Summary

A final snapshot of the clean dataset before export.
All values here are computed dynamically from the data.

In [ ]:
print("=" * 55)
print("  CLEAN DATASET SUMMARY")
print("=" * 55)
print(f"  Total posts:          {len(df)}")
print(f"  Date range:           {df['date'].min()}  →  {df['date'].max()}")
print(f"  Post types:           {dict(df['post_type'].value_counts())}")
print(f"  Post categories:      {dict(df['post_category'].value_counts())}")
print()
print(f"  Avg engagement rate:  {df['engagement_rate'].mean():.1%}")
print(f"  Median impressions:   {df['impressions'].median():,.0f}")
print(f"  Top views (single):   {df['views'].max():,}")
print()
print(f"  Avg text length:      {df['text_length_words'].mean():.0f} words")
print("=" * 55)

  CLEAN DATASET SUMMARY
  Total posts:          99
  Date range:           2025-06-04  →  2026-03-19
  Post types:           {'Photo': np.int64(77), 'Content': np.int64(15), 'Reel': np.int64(7)}
  Post categories:      {'Algorithm Pushed': np.int64(27), 'Audience Favorite': np.int64(27), 'Viral': np.int64(23), 'Low Performance': np.int64(22)}

  Avg engagement rate:  11.6%
  Median impressions:   844
  Top views (single):   16,999

  Avg text length:      999 characters
  Posts with no title:  2


## 8. Export

Save the clean dataframe to CSV. All subsequent notebooks load from this file —
never from the raw Facebook export directly.

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"✓ Exported {len(df)} posts to {OUTPUT_FILE}")
print(f"  Columns: {df.columns.tolist()}")

✓ Exported 99 posts to ../data/clean_posts.csv
  Columns: ['post_id', 'page_id', 'page_name', 'title', 'duration_sec_', 'publish_time', 'permalink', 'post_type', 'data_comment', 'date', 'comments', 'distribution', 'impressions', 'interactions', 'reactions', 'saves', 'shares', 'views', 'viewers', 'net_follows', 'average_seconds_viewed', 'seconds_viewed', 'time', 'hour', 'weekday', 'post_origin', 'text_length', 'engagement_rate', 'repeat_view_rate', 'view_through_rate', 'post_category']
